# EDM U-Net with Covariance-Weighted Tikhonov (training-time)

**Purpose.** Bring the two threads together: the closed-form GMM study
(`gmm_tikhonov_variants_comparison.ipynb`) showed covariance-weighted Tikhonov is
*scale-selective* — it de-memorizes the fine band at $c$ ~100x smaller than plain Tikhonov
while leaving the coarse band memorized. This notebook tests whether a **trained U-Net**
reproduces that behaviour when the same regularizer is applied in the loss.

**The penalty** (`train_edm(..., c_tikhonov_cov=c)`), per Fourier mode $k$:

$$ c\,\lambda_{\mathrm{EDM}}(\sigma)\;\overline{\left|\widehat{(D_\theta - x)}_k\right|^2 \big/ \big(\sigma^2\lambda(k)\big)} $$

Both the denoising term (Parseval, orthonormal FFT) and this penalty are mode-diagonal, so
the minimization decouples per mode and the stationary score is

$$ s^*_k \;=\; \frac{s_{\mathrm{true},k}}{1 + c/(\sigma^2\lambda(k))} \;=\; \frac{(\mathbb{E}[x_0|x]-x)_k}{\sigma^2 + c/\lambda(k)}, $$

i.e. exactly the denominator of `GMM_score_CovarianceTikhonov`, and the training-time
analogue of Baptista et al. §5.1 with matrix weight $\Gamma(t) = \tfrac{c}{\sigma^2}\Sigma^{-1}$.
(Both stationary points are pinned by tests in `src/tests/test_consolidation.py`.)

We compare three training runs at matched $c$: **unregularized**, **isotropic**
(`c_tikhonov`), and **covariance-weighted** (`c_tikhonov_cov`), then sample each with the
plain score $(D_\theta-x)/\sigma^2$ — the regularization is already baked into the weights,
so applying it again at sampling time would double-count.

**Runtime:** 3 configs x |C_VALUES| trainings. Set `SMOKE = True` for a minutes-long check.

In [ ]:
import sys, os, math, time
import numpy as np
import torch
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import diffusion_score_models as score_models
from multiband_data_utils import generate_multiband_dataset_postmask
from memorization_metrics import RingMetricContext, radial_power_spectrum
from edm import EDMPrecond, EDMScoreWrapper, train_edm, hermitian_symmetrize
from unet import SmallUNet
from device_utils import resolve_device

DEVICE = resolve_device()   # cuda > mps > cpu; use resolve_device("cpu") on the personal laptop
print(f'device: {DEVICE}')

In [ ]:
# -- Data (identical config to the earlier memorization notebooks) --
components = [
    {"name": "coarse", "length_scale": 2.0,  "s": 2.0, "sigma_sq": 1.0, "band": (0.5, 4.0)},
    {"name": "mid1",   "length_scale": 6.0,  "s": 2.0, "sigma_sq": 1.0, "band": (4.0, 10.0)},
    {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0, "band": (10.0, 18.0)},
    {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0, "band": (18.0, 32.0)},
]
result = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=128, components=components,
    weights=[1.0, 0.8, 0.8, 1.2], seed=42, normalize=True,
)
bands = result.get('bands', {c['name']: c['band'] for c in components})
N = 128
N_TRAIN = 8
x_all = result['combined']
x_train = x_all[:N_TRAIN].to(DEVICE)
train_flat = x_train.reshape(N_TRAIN, -1)

ctx = RingMetricContext(N, bands, device=DEVICE)

# per-mode data variance lambda(k) (Hermitian-symmetric by construction)
imgs_c = x_train - x_train.mean(dim=0, keepdim=True)
lam = (torch.fft.fft2(imgs_c, dim=(-2, -1), norm='ortho').abs() ** 2).mean(dim=0)
lam = hermitian_symmetrize(lam)
print(f'mean(lambda) = {lam.mean():.4f}')
for name in bands:
    m = ctx.ring_masks[ctx.band_rings[name]].sum(dim=0).bool()
    print(f'  band {name:>6}: mean lambda = {lam[m].mean():.3e}')

In [ ]:
# -- Sampling / evaluation helpers (matched sampler, sigma_max=10) --
VE_SAMPLE = score_models.VE_EDM(sigma_min=0.002, sigma_max=10.0)
N_GEN, N_SDE_STEPS, N_RAND_REF, LATENT_SEED = 16, 1000, 32, 42

@torch.no_grad()
def evaluate(precond):
    # sample with the PLAIN score: the regularization is already in the weights,
    # so c_tikhonov=0 here avoids double-counting it
    wrapper = EDMScoreWrapper(precond, VE_SAMPLE.marginal_prob_std, N, c_tikhonov=0.0).to(DEVICE)
    torch.manual_seed(LATENT_SEED)
    latents = torch.randn(N_GEN, N*N, device=DEVICE)
    x_gen = VE_SAMPLE.SDEsampler(wrapper, latents, num_steps=N_SDE_STEPS).reshape(N_GEN, N, N)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'mean_ratio': m['mean_ratio'].cpu(),
        'gen_spectrum': radial_power_spectrum(x_gen, ctx.ring_masks).cpu(),
        'samples': x_gen[:4].cpu(),
    }

results_dir = os.path.join(repo_root, 'results', 'data')
fig_dir = os.path.join(repo_root, 'results', 'figures')
os.makedirs(results_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
# -- Sweep configuration --
SMOKE = False    # True: tiny end-to-end run to verify the pipeline

TOTAL_STEPS = 20000
C_VALUES = [1e-3, 1e-2, 0.1, 1.0]
BATCH_SIZE = 8

if SMOKE:
    TOTAL_STEPS = 200
    C_VALUES = [1e-2]
    N_SDE_STEPS, N_GEN = 50, 4

ckpt_path = os.path.join(results_dir, 'edm_unet_covariance_tikhonov.pt')
print(f'steps={TOTAL_STEPS}, c values={C_VALUES}')

In [ ]:
# -- Train: unregularized baseline + isotropic and covariance at each c --
runs = {}
if os.path.exists(ckpt_path):
    runs = torch.load(ckpt_path, map_location='cpu', weights_only=False).get('runs', {})
    print(f'loaded existing runs: {sorted(runs.keys())}')

def train_one(tag, **reg_kwargs):
    if tag in runs:
        print(f'{tag}: already trained, skipping')
        return
    print(f'===== {tag} =====')
    t0 = time.time()
    saved = train_edm(train_flat, grid_size=N, total_steps=TOTAL_STEPS,
                      checkpoint_at=[TOTAL_STEPS], base_channels=16, emb_dim=64,
                      lr=1e-3, batch_size=BATCH_SIZE, seed=0, device=DEVICE,
                      UNetClass=SmallUNet, **reg_kwargs)
    p = saved[TOTAL_STEPS]
    runs[tag] = {'state_dict': {k: v.cpu() for k, v in p.state_dict().items()},
                 'sigma_data': p.sigma_data}
    torch.save({'runs': runs, 'lam': lam.cpu(), 'c_values': C_VALUES,
                'total_steps': TOTAL_STEPS, 'n_train': N_TRAIN}, ckpt_path)
    print(f'  {time.time()-t0:.0f}s')

train_one('unregularized')
for c_val in C_VALUES:
    train_one(f'iso_c{c_val:g}', c_tikhonov=c_val)
    train_one(f'cov_c{c_val:g}', c_tikhonov_cov=c_val, cov_spectrum=lam)

In [ ]:
# -- Evaluate every run + GMM closed-form references --
evals = {}
for tag, entry in runs.items():
    unet = SmallUNet(base_channels=16, emb_dim=64).to(DEVICE)
    precond = EDMPrecond(unet, sigma_data=entry['sigma_data']).to(DEVICE)
    precond.load_state_dict(entry['state_dict'])
    precond.eval()
    t0 = time.time()
    evals[tag] = evaluate(precond)
    print(f"{tag:>16}: coarse={evals[tag]['coarse_score']:.4f} "
          f"fine={evals[tag]['fine_score']:.4f}  ({time.time()-t0:.0f}s)")

# closed-form GMM counterparts at the same c, for reference
gmm_evals = {}
mpm, mps_, dc = VE_SAMPLE.marginal_prob_mean, VE_SAMPLE.marginal_prob_std, VE_SAMPLE.diffusion_coeff

@torch.no_grad()
def eval_score_fn(score_fn):
    torch.manual_seed(LATENT_SEED)
    latents = torch.randn(N_GEN, N*N, device=DEVICE)
    x_gen = VE_SAMPLE.SDEsampler(score_fn, latents, num_steps=N_SDE_STEPS).reshape(N_GEN, N, N)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {'coarse_score': m['coarse_score'].mean().item(),
            'fine_score': m['fine_score'].mean().item(),
            'mean_ratio': m['mean_ratio'].cpu()}

gmm_evals['unregularized'] = eval_score_fn(
    score_models.GMM_score(train_flat, mpm, mps_).to(DEVICE))
for c_val in C_VALUES:
    gmm_evals[f'iso_c{c_val:g}'] = eval_score_fn(
        score_models.GMM_score_TikhonovRegularized(train_flat, mpm, mps_, dc,
                                                   constant=c_val).to(DEVICE))
    gmm_evals[f'cov_c{c_val:g}'] = eval_score_fn(
        score_models.GMM_score_CovarianceTikhonov(train_flat, mpm, mps_, N,
                                                  constant=c_val, spectrum=lam).to(DEVICE))
for tag, e in gmm_evals.items():
    print(f"GMM {tag:>16}: coarse={e['coarse_score']:.4f} fine={e['fine_score']:.4f}")

In [ ]:
# -- Plot: does the U-Net inherit the scale-selectivity of the closed form? --
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), sharey=True)
c_plot = C_VALUES
for ax, band in zip(axes, ['coarse', 'fine']):
    for kind, color in [('iso', 'tab:gray'), ('cov', 'tab:green')]:
        unet_y = [evals[f'{kind}_c{c:g}'][f'{band}_score'] for c in C_VALUES]
        gmm_y = [gmm_evals[f'{kind}_c{c:g}'][f'{band}_score'] for c in C_VALUES]
        ax.plot(c_plot, unet_y, marker='o', color=color, lw=1.8, label=f'U-Net {kind}')
        ax.plot(c_plot, gmm_y, marker='.', color=color, lw=1.0, ls=':', label=f'GMM {kind}')
    ax.axhline(evals['unregularized'][f'{band}_score'], color='tab:red', lw=1.0, ls='--',
               label='U-Net unregularized')
    ax.axhline(1.0, color='gray', lw=0.8, ls='--')
    ax.set_xscale('log')
    ax.set_xlabel('c')
    ax.set_title(f'{band} band')
axes[0].set_ylabel('band score (<1 = memorized)')
axes[1].legend(fontsize=8)
fig.suptitle('Training-time Tikhonov in the U-Net vs the closed-form GMM (dotted)', y=1.04)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'unet_covariance_tikhonov_band_scores.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- Per-wavenumber view + save --
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
kc = ctx.k_centers.cpu().numpy()
cmap = plt.cm.viridis
for ax, kind, title in [(axes[0], 'iso', 'isotropic (training-time)'),
                        (axes[1], 'cov', 'covariance-weighted (training-time)')]:
    ax.plot(kc, evals['unregularized']['mean_ratio'].numpy(), color='tab:red', lw=1.8,
            label='unregularized')
    for j, c_val in enumerate(C_VALUES):
        ax.plot(kc, evals[f'{kind}_c{c_val:g}']['mean_ratio'].numpy(),
                color=cmap(j / max(len(C_VALUES)-1, 1)), lw=1.5, label=f'c={c_val:g}')
    for bname, color in [('coarse', 'tab:blue'), ('fine', 'tab:red')]:
        lo, hi = bands[bname]
        ax.axvspan(lo, hi, color=color, alpha=0.08)
    ax.axhline(1.0, color='gray', lw=0.8, ls='--')
    ax.set_xlim(0, 45); ax.set_xlabel('wavenumber k'); ax.set_title(title)
axes[0].set_ylabel('mean ratio'); axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'unet_covariance_tikhonov_per_k.png'),
            dpi=150, bbox_inches='tight')
plt.show()

torch.save({'runs': runs, 'evals': evals, 'gmm_evals': gmm_evals, 'lam': lam.cpu(),
            'c_values': C_VALUES, 'total_steps': TOTAL_STEPS, 'n_train': N_TRAIN,
            'k_centers': ctx.k_centers.cpu(), 'bands': bands,
            'note': ('EDM UNet trained with training-time Tikhonov: isotropic (c_tikhonov) vs '
                     'covariance-weighted (c_tikhonov_cov, denominator sigma^2 + c/lambda(k)). '
                     'Sampled with the plain score (c=0 wrapper) to avoid double-counting. '
                     'GMM closed-form counterparts at matched c for reference.')},
           ckpt_path)
print(f'saved -> {ckpt_path}')

## Notes

- **What to look for:** in the closed form, `cov` lifts the fine band toward 1 while holding
  the coarse band down; `iso` moves both together. If the U-Net curves track their GMM
  counterparts, the scale-selective regularizer transfers to the learned setting — the
  paper-ready synthesis of both threads.
- **If they don't track:** the U-Net's own inductive bias may dominate at this capacity /
  training length (consistent with the mechanism notebook, where the unregularized U-Net
  already fails to memorize from pure noise). In that case, run this at a training length
  where the *unregularized* baseline does memorize (see
  `edm_unet_memorization_transition.ipynb`) — regularization can only be shown to remove
  memorization that was there to begin with.
- Sampling deliberately uses the plain score. Both the training penalty and the
  `EDMScoreWrapper(c_tikhonov=...)` knob implement the same $\sigma^2 \to \sigma^2 + c$
  shift, so using both would apply it twice.